In [1]:
import pandas as pd 
import os 

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

import seaborn as sns

from datasets import load_dataset, load_from_disk
import numpy as np
import json

In [2]:
pwd

'/mnt/home/al2644/research/projects/perturb-r/notebook'

In [5]:
cd /mnt/home/al2644/research/projects/rlvr

/mnt/home/al2644/research/projects/perturb-r


In [6]:
from utils import chunk_r 

# Check the Dataset

In [13]:
path = "data/countdown_train_stage2_level5_100K/"
dataset = load_from_disk(path)

In [16]:
dataset['test'][0]

{'nums': [61, 11, 50, 60, 71],
 'target': 133,
 'example_solution': '(((71+61)+50)-(60-11))',
 'problem': 'Please reason and answer the following question. Using the numbers [61, 11, 50, 60, 71], create an equation that equals 133. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.',
 'level': 5,
 'task': 'countdown'}

# Training Data

In [18]:
cd /mnt/home/al2644/research/projects/rlvr

/mnt/home/al2644/research/projects/rlvr


In [21]:
stage1_df = pd.read_parquet("data/train/countdown_level5_35k/test.parquet")
stage2_df = pd.read_parquet("data/train/countdown_level5_50k/test.parquet")

In [22]:
stage1_df

,data_source,prompt,ability,reward_model,extra_info
0,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [56, 46, 14, 36, ...","{'index': 0, 'level': '6', 'split': 'test'}"
1,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [2, 76, 58, 8, 53...","{'index': 1, 'level': '6', 'split': 'test'}"
2,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [32, 12, 23, 86, ...","{'index': 2, 'level': '6', 'split': 'test'}"
3,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [6, 17, 92, 23, 7...","{'index': 3, 'level': '6', 'split': 'test'}"
4,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [23, 43, 33, 33, ...","{'index': 4, 'level': '6', 'split': 'test'}"
...,...,...,...,...,...
295,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [95, 38, 13, 72, ...","{'index': 295, 'level': '7', 'split': 'test'}"
296,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [51, 100, 99, 91,...","{'index': 296, 'level': '7', 'split': 'test'}"
297,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [44, 2, 3, 41, 85...","{'index': 297, 'level': '7', 'split': 'test'}"
298,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [55, 20, 64, 3, 4...","{'index': 298, 'level': '7', 'split': 'test'}"


In [23]:
stage2_df

,data_source,prompt,ability,reward_model,extra_info
0,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [50, 77, 64, 67, ...","{'index': 0, 'level': '6', 'split': 'test'}"
1,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [88, 84, 11, 71, ...","{'index': 1, 'level': '6', 'split': 'test'}"
2,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [35, 48, 94, 46, ...","{'index': 2, 'level': '6', 'split': 'test'}"
3,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [18, 58, 64, 42, ...","{'index': 3, 'level': '6', 'split': 'test'}"
4,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [37, 18, 51, 39, ...","{'index': 4, 'level': '6', 'split': 'test'}"
...,...,...,...,...,...
295,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [95, 38, 13, 72, ...","{'index': 295, 'level': '7', 'split': 'test'}"
296,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [51, 100, 99, 91,...","{'index': 296, 'level': '7', 'split': 'test'}"
297,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [44, 2, 3, 41, 85...","{'index': 297, 'level': '7', 'split': 'test'}"
298,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [55, 20, 64, 3, 4...","{'index': 298, 'level': '7', 'split': 'test'}"


# Corruption Data

In [26]:
cd ../perturb-r/

/mnt/home/al2644/research/projects/perturb-r


In [33]:
path = "./results/countdown_train_stage2_level5_100K/corrupt_numbers/Qwen2.5-3B-countdown-level4-5-stage1_rl.pickle"
df = pd.read_pickle(path)

In [34]:
df.groupby("")

,nums,target,example_solution,problem,level,task,original_response,reasoning_chunks,start_pos,end_pos,prefix_reasoning,corrupted_reasoning,replacement,prompt,post_corruption_response,still_correct,if_answer
0,"[52, 80, 79, 65, 30]",148,(((65-79)+(80+52))+30),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.22,0.52,"<think>\nOkay, let's see. I need to use each o...","Alternatively, maybe multiply some numbers. Le...","{'65': '5', '42': '442', '28': '8', '132': '12...",<|im_start|>system\nYou are a helpful assistan...,"Not helpful.\n\nAlternatively, 552 - 800 is -...",1.0,True
1,"[52, 80, 79, 65, 30]",148,(((65-79)+(80+52))+30),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.00,0.30,,"<think>\nOkay, let's see. I need to use each o...","{'1': '0', '65': '5', '42': '4', '28': '2', '1...",<|im_start|>system\nYou are a helpful assistan...,799 + 330 = 1129. Then 1129 plus 8 is 1137. T...,0.0,False
2,"[52, 80, 79, 65, 30]",148,(((65-79)+(80+52))+30),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.00,0.30,,"<think>\nOkay, let's see. I need to use each o...","{'1': '7', '65': '5', '42': '24', '28': '2', '...",<|im_start|>system\nYou are a helpful assistan...,800 + 522 = 1322. Then 1322 + 97 = 1419. Then...,0.0,True
3,"[52, 80, 79, 65, 30]",148,(((65-79)+(80+52))+30),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.22,0.52,"<think>\nOkay, let's see. I need to use each o...","Alternatively, maybe multiply some numbers. Le...","{'65': '56', '42': '24', '28': '228', '132': '...",<|im_start|>system\nYou are a helpful assistan...,"Not enough.\n\nAlternatively, maybe 79 - 30 =...",1.0,True
4,"[62, 6, 90, 59, 68]",13,(((90-6)-62)-(68-59)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.00,0.31,,"<think>\nOkay, let's see. I need to use each o...","{'56': '566', '25': '5', '3': '0', '75': '57',...",<|im_start|>system\nYou are a helpful assistan...,Maybe 566 - 3 = 563. Not helpful.\n\nAlternat...,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,"[1, 19, 10, 44, 44]",96,(((44-10)+44)+(19-1)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use the nu...","[<think>\nOkay, let's see. I need to use the n...",0.00,0.33,,"<think>\nOkay, let's see. I need to use the nu...","{'1': '9', '10': '1', '9': '7', '44': '4', '8'...",<|im_start|>system\nYou are a helpful assistan...,might not be directly applicable.\n\nAlternati...,0.0,False
396,"[84, 19, 14, 46, 48]",77,((46+14)-((19-84)+48)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use the nu...","[<think>\nOkay, let's see. I need to use the n...",0.00,0.31,,"<think>\nOkay, let's see. I need to use the nu...","{'48': '4', '84': '4', '67': '76', '99': '9', ...",<|im_start|>system\nYou are a helpful assistan...,Maybe another approach.\n\nAnother idea: mayb...,0.0,True
397,"[84, 19, 14, 46, 48]",77,((46+14)-((19-84)+48)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use the nu...","[<think>\nOkay, let's see. I need to use the n...",0.23,0.54,"<think>\nOkay, let's see. I need to use the nu...","Alternatively, maybe adding some numbers. Let'...","{'71': '7', '94': '9', '48': '488', '99': '999...",<|im_start|>system\nYou are a helpful assistan...,"Hmm.\n\nAlternatively, maybe 84 - 46 = 38. Th...",1.0,True
398,"[84, 19, 14, 46, 48]",77,((

In [32]:
100 * 0.2

20.0